In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.azure_sql import (
    get_engine,
    test_connection,
    read_sql,
    write_dataframe
)

In [3]:
engine = get_engine()
test_connection(engine)

print("Azure SQL: OK")

Azure SQL: OK


In [4]:
fact_gl = read_sql(
    """
    SELECT *
    FROM silver.fact_gl
    """,
    engine
)

dim_account = read_sql(
    """
    SELECT *
    FROM silver.dim_account
    """,
    engine
)

print("GL rows:", len(fact_gl))
print("Accounts:", len(dim_account))

GL rows: 1116
Accounts: 89


In [5]:
gl = fact_gl.merge(
    dim_account[
        [
            "account_id",
            "account_type",
            "account_subtype",
            "classification"
        ]
    ],
    on="account_id",
    how="left"
)

In [6]:
pnl = gl[
    gl["classification"].isin(
        ["Revenue", "Expense"]
    )
].copy()

In [7]:
pnl["actual_amount"] = pnl.apply(
    lambda x:
        -x["signed_amount"]
        if x["classification"] == "Revenue"
        else x["signed_amount"],
    axis=1
)

In [8]:
pnl_monthly_actual = (
    pnl
    .groupby(
        [
            "year",
            "month",
            "year_month",
            "account_id",
            "account_name",
            "account_type",
            "account_subtype",
            "classification"
        ],
        dropna=False
    )["actual_amount"]
    .sum()
    .reset_index()
)

In [9]:
display(pnl_monthly_actual.head(20))

,year,month,year_month,account_id,account_name,account_type,account_subtype,classification,actual_amount
0,2023,9,2023-09,1,Services,Income,ServiceFeeIncome,Revenue,528181.00
1,2023,9,2023-09,10,Dues & Subscriptions,Expense,DuesSubscriptions,Expense,14190.55
2,2023,9,2023-09,11,Insurance,Expense,Insurance,Expense,14000.00
3,2023,9,2023-09,12,Legal & Professional Fees,Expense,LegalProfessionalFees,Expense,4366.32
4,2023,9,2023-09,13,Meals and Entertainment,Expense,EntertainmentMeals,Expense,818.69
5,2023,9,2023-09,14,Miscellaneous,Other Expense,OtherMiscellaneousExpense,Expense,654.95
6,2023,9,2023-09,15,Office Expenses,Expense,OfficeGeneralAdministrativeExpenses,Expense,10091.58
7,2023,9,2023-09,16,Promotional,Expense,AdvertisingPromotional,Expense,5332.26
8,2023,9,2023-09,17,Rent or Lease,Expense,RentOrLeaseOfBuildings,Expense,42000.00
9,2023,9,2023-09,21,Taxes & Licenses,Expense,TaxesPaid,Expense,12000.00


In [10]:
print(
    "Months:",
    pnl_monthly_actual["year_month"].nunique()
)

print(
    "From:",
    pnl_monthly_actual["year_month"].min()
)

print(
    "To:",
    pnl_monthly_actual["year_month"].max()
)

print(
    "Total revenue:",
    pnl_monthly_actual.loc[
        pnl_monthly_actual["classification"] == "Revenue",
        "actual_amount"
    ].sum()
)

print(
    "Total expenses:",
    pnl_monthly_actual.loc[
        pnl_monthly_actual["classification"] == "Expense",
        "actual_amount"
    ].sum()
)

Months: 36
From: 2023-09
To: 2026-08
Total revenue: 42915656.31
Total expenses: 48034298.67


In [11]:
write_dataframe(
    pnl_monthly_actual,
    table="pnl_monthly_actual",
    schema="gold",
    if_exists="replace",
    engine=engine
)

print("Gold load: OK")

Gold load: OK


In [12]:
check = read_sql(
    """
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT year_month) AS months,
        MIN(year_month) AS first_month,
        MAX(year_month) AS last_month
    FROM gold.pnl_monthly_actual
    """,
    engine
)

display(check)

,rows,months,first_month,last_month
0,828,36,2023-09,2026-08
